In [6]:
import pandas as pd
import numpy as np

In [7]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
universal_dataset = pd.read_parquet(
    "C:/Users/ujjwa/Desktop/amazon_sentiment_analysis/data/processed/universal_dataset.parquet",
    engine="pyarrow"
)

In [19]:
df = pd.DataFrame(universal_dataset)
print(df.shape)
print(df.columns)
df.sample(5)

(100000, 11)
Index(['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id',
       'timestamp', 'helpful_vote', 'verified_purchase', 'category'],
      dtype='str')


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,category
64128,4.0,Satisfied,Great product,[],B01DB1SOMK,B0C48B1TR1,AG3S5WUSQCM7LNBNZ3GCTWOTTEVA,1547973618691,1,True,Handmade_Products
49728,5.0,Nice design,[[VIDEOID:91267c8d71d47b46c273fd18b903b5da]] T...,[{'small_image_url': 'https://m.media-amazon.c...,B0B8SKVWW2,B0B9LWVRG1,AFGDRVPCP742YM5MMLFIKZCGNNRQ,1661559485790,0,True,Electronics
69944,5.0,cat print,Love love all 5 that I have purchased as gifts...,[],B01HSOMK0S,B01HSOMK0S,AHWWKNI3CITAVCZVFFQ6TKKZ4VMQ,1489515371000,1,True,Handmade_Products
89359,5.0,Variety of herbs,I didn't expect so many herbs<br />The package...,[],B08NXFDQ1L,B08NXFDQ1L,AETW63I2QJDXWB6VGN5UBTLHIV2A,1644079947341,0,True,Health_and_Personal_Care
51346,5.0,Five Stars,"Good price, good product.",[],B011S6YKYQ,B011S6YKYQ,AFVVAWYOXQK63S7ZDK5EZGNM4LSA,1518994696148,0,True,Electronics


In [20]:
df.isnull().sum()

rating               0
title                0
text                 0
images               0
asin                 0
parent_asin          0
user_id              0
timestamp            0
helpful_vote         0
verified_purchase    0
category             0
dtype: int64

In [21]:
df.describe()

,rating,timestamp,helpful_vote
count,100000.00000,1.000000e+05,100000.000000
mean,4.41173,1.548512e+12,1.230820
std,1.13432,9.700586e+10,9.221351
min,1.00000,9.049815e+11,0.000000
25%,4.00000,1.490279e+12,0.000000
50%,5.00000,1.571017e+12,0.000000
75%,5.00000,1.620922e+12,1.000000
max,5.00000,1.679872e+12,777.000000


In [22]:
#Exclude columns with unhashable list types (like 'images') to check for duplicates
df.drop(columns = ['images']).duplicated().sum()

np.int64(221)

In [23]:
df[df.duplicated(subset = ["text", "user_id", "parent_asin"], keep = False )]

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,category
1273,5.0,Piano Solo,I am singing solo for Christmas Cantata...this...,[],B0012A4SFA,B0012A4SFA,AEQ4BRZQAEE7UKKOPVTM2EQVGV3A,1384398192000,1,True,Digital_Music
1274,5.0,Piano Solo,I am singing solo for Christmas Cantata...this...,[],B0012A4SFA,B0012A4SFA,AEQ4BRZQAEE7UKKOPVTM2EQVGV3A,1384398192000,1,True,Digital_Music
1340,5.0,great price,great product. great price,[],B002LVGU82,B002LVGU82,AHKQKE46BOZQDE6TIKGSOUS3NEYQ,1548159682656,0,True,Digital_Music
1341,5.0,great price,great product. great price,[],B002LVGU82,B002LVGU82,AHKQKE46BOZQDE6TIKGSOUS3NEYQ,1548159682656,0,True,Digital_Music
1427,5.0,Scurve E. Dogs...,"Scuttle me bones, me buckoes, 'ere be more sal...",[],B00A82J4X4,B00A82J4X4,AEK27DQ54G7PIWW627UHT72TFX4A,1359670654000,1,False,Digital_Music
...,...,...,...,...,...,...,...,...,...,...,...
98674,5.0,Great item,Started working on a go bag. Holds more than y...,[],B01LW81EBO,B07VX81L84,AHH645F7BT3BTF7YCMFD5W4EXTGQ,1610835944368,0,True,Health_and_Personal_Care
98816,5.0,Great,"Strong vinegar, delisted and using as househol...",[],B08NPN1PG3,B0BR8WTXV5,AFPQRHHT4J5JAKKG4HGRCQNMDPIQ,1617632842246,2,True,Health_and_Personal_Care
98820,5.0,Great,"Strong vinegar, delisted and using as househol...",[],B08NPN1PG3,B0BR8WTXV5,AFPQRHHT4J5JAKKG4HGRCQNMDPIQ,1617632842246,2,True,Health_and_Personal_Care
99155,3.0,A bit disappointing,"Meh. Plates are flimsy, utensils are tiny, and...",[],B07VTNLRJC,B07VTNLRJC,AE2VA3TLLKW33LOGI6KMXC6AROUA,1583358635711,0,True,Health_and_Personal_Care


In [24]:
df = df.drop_duplicates(subset = ["text", "user_id", "parent_asin"],keep = False)

# Feature Engineering

In [25]:
#1. Remove HTML Tags
def remove_html(tags):
    import re
    clean = re.compile('<.*?.')
    return re.sub(clean, '', tags)

In [26]:
df['text'][20]

'The follow-up to his "The Street Was Always There" collection, and basically, it\'s more of the same.  A journeyman\'s collection of folk and folk-rock covers (mostly) delivered in Andersen\'s smooth, thoughtful style.  If you were a fan of \'Street,\' then "Waves" will be right in your wheelhouse.  Three-and-one-half stars and a mildly enthusiastic recommendation from this longtime fan.  I much prefer him in singer / songwriter mode rather than as an interpreter of other writers\' works.'

In [27]:
#2. Convert everything to lower
def lower_case(text):
    return text.lower()


In [28]:
df['text'][20]

'The follow-up to his "The Street Was Always There" collection, and basically, it\'s more of the same.  A journeyman\'s collection of folk and folk-rock covers (mostly) delivered in Andersen\'s smooth, thoughtful style.  If you were a fan of \'Street,\' then "Waves" will be right in your wheelhouse.  Three-and-one-half stars and a mildly enthusiastic recommendation from this longtime fan.  I much prefer him in singer / songwriter mode rather than as an interpreter of other writers\' works.'

In [29]:
#Removing special char
def remove_special(text):
    if not isinstance(text, str):
        return ""

    x = ""
    for i in text:
        if i.isalnum():
            x += i
        else:
            x += " "

    return x

In [30]:
df['text'][20]

'The follow-up to his "The Street Was Always There" collection, and basically, it\'s more of the same.  A journeyman\'s collection of folk and folk-rock covers (mostly) delivered in Andersen\'s smooth, thoughtful style.  If you were a fan of \'Street,\' then "Waves" will be right in your wheelhouse.  Three-and-one-half stars and a mildly enthusiastic recommendation from this longtime fan.  I much prefer him in singer / songwriter mode rather than as an interpreter of other writers\' works.'

In [31]:
#Remove stopwords
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ujjwa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [32]:
from nltk.corpus import stopwords

In [33]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    x = []

    for i in text.split():
        if i not in stop_words:
            x.append(i)

    return " ".join(x)

In [34]:
df['text'][20]

'The follow-up to his "The Street Was Always There" collection, and basically, it\'s more of the same.  A journeyman\'s collection of folk and folk-rock covers (mostly) delivered in Andersen\'s smooth, thoughtful style.  If you were a fan of \'Street,\' then "Waves" will be right in your wheelhouse.  Three-and-one-half stars and a mildly enthusiastic recommendation from this longtime fan.  I much prefer him in singer / songwriter mode rather than as an interpreter of other writers\' works.'

In [35]:
#Perform stemming
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [36]:
def stem_words(text):
    y = []

    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)

In [37]:
df['text'][20]

'The follow-up to his "The Street Was Always There" collection, and basically, it\'s more of the same.  A journeyman\'s collection of folk and folk-rock covers (mostly) delivered in Andersen\'s smooth, thoughtful style.  If you were a fan of \'Street,\' then "Waves" will be right in your wheelhouse.  Three-and-one-half stars and a mildly enthusiastic recommendation from this longtime fan.  I much prefer him in singer / songwriter mode rather than as an interpreter of other writers\' works.'

In [38]:
def join_back(text):
    return "".join(text)

In [39]:
join_back(['d', 'o', 'l', 'l', 'a', 'r', ' ', 'm', 'a', 'n', 'y'])

'dollar many'

In [40]:
df['text'][20]

'The follow-up to his "The Street Was Always There" collection, and basically, it\'s more of the same.  A journeyman\'s collection of folk and folk-rock covers (mostly) delivered in Andersen\'s smooth, thoughtful style.  If you were a fan of \'Street,\' then "Waves" will be right in your wheelhouse.  Three-and-one-half stars and a mildly enthusiastic recommendation from this longtime fan.  I much prefer him in singer / songwriter mode rather than as an interpreter of other writers\' works.'

In [41]:
def clean_text(text):
    text = remove_html(text)
    text = lower_case(text)
    text = remove_special(text)
    text = remove_stopwords(text)
    text = stem_words(text)
    text = join_back(text)

    return text

# Data preprocessing and transformation

In [42]:
#Create sentiment labels from ratings
def get_sentiment(rating):
    if rating < 3:
        return "negative"
    elif rating == 3:
        return "neutral"
    else:
        return "positive"

df['sentiment'] = df['rating'].apply(get_sentiment)

In [43]:
#Testinf sentiment labeling
df['sentiment'][20]

'positive'

In [44]:
%pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [45]:
def preprocess_text(x):
    return [clean_text(text) for text in x]

In [46]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler

from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import ComplementNB

pipe = Pipeline([
    ("clean_text", FunctionTransformer(
        preprocess_text,
        validate=False
    )),

    ("vectorizer", TfidfVectorizer(max_features=10000)),

    ("sampler", RandomOverSampler(
        sampling_strategy={
            "negative": 33506,
            "neutral": 33506
        },
        random_state=42
    )),

    ("classifier", ComplementNB())
])

In [47]:
#Input features and target
X = df['text']
y = df["rating"].apply(get_sentiment)

In [48]:
#Split the data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42,stratify=y)

In [49]:
print("training_shape", X_train.shape)
print("testing_shape",X_test.shape)

training_shape (79567,)
testing_shape (19892,)


In [50]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__alpha": [0.01, 0.1, 0.5, 1.0, 2.0]
}

grid = GridSearchCV(
    pipe,
    param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV Macro F1:", grid.best_score_)

Best Parameters: {'classifier__alpha': 2.0}
Best CV Macro F1: 0.532591651349812


In [51]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7714156444801931

Classification Report:
              precision    recall  f1-score   support

    negative       0.40      0.67      0.50      1870
     neutral       0.19      0.38      0.25      1262
    positive       0.96      0.81      0.88     16760

    accuracy                           0.77     19892
   macro avg       0.52      0.62      0.55     19892
weighted avg       0.86      0.77      0.80     19892


Confusion Matrix:
[[ 1256   340   274]
 [  432   483   347]
 [ 1441  1713 13606]]


In [52]:
import os

os.makedirs("../models", exist_ok=True)

In [54]:
import cloudpickle

with open("../models/model.pkl", "wb") as f:
    cloudpickle.dump(best_model, f)

print("Model saved successfully!")

Model saved successfully!
